# 02 Build the matched dataset

Stations from Frost, element ids confirmed, observations downloaded and quality filtered, then every field file matched to every station and written as `matched.parquet` with a content-hashed manifest.

## What this notebook does and does not show

Does: select and freeze the station list (Bergen northwards, four counties), resolve the Frost element ids and flag the unconfirmed ones, fetch `t2m`, `ws10` and `msl` observations with the raw quality codes kept, match grid to station with both interpolation methods, add the persistence and climatology baselines, apply the pairwise-complete rule, and write the dataset.

Does not: score anything. Precipitation (`tp24`) is disabled in the default config and is not fetched here.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from afv.config import enabled_variables, load_config

CONFIG = Path(os.environ.get("AFV_CONFIG", "configs/default.yaml"))
cfg = load_config(CONFIG)
print(CONFIG, "->", cfg["name"])
print("enabled variables:", enabled_variables(cfg))
print("stations csv:", cfg.get_path("stations_csv"))

## Stations

The next cell uses the network: one Frost `/sources` request per county. Frost needs `FROST_CLIENT_ID` in the environment (see notebook 00). The counties are Vestland, Trøndelag, Nordland and Troms; the latitude floor 60.39 is Bergen.

In [ ]:
from afv.stations import fetch_sources, available_series, select_stations, freeze
from afv.fetch_obs import ELEMENTS

st = cfg["stations"]
counties = ["Vestland", "Trøndelag", "Nordland", "Troms"]
src = fetch_sources(None, counties=counties, bbox=st.get("bbox"))
print(len(src), "SensorSystem sources returned")
src.head()

The next cell uses the network: `/observations/availableTimeSeries` in chunks of 50 stations, to learn which stations report which elements and with what siting (`exposureCategory` 1 best to 5) and sensor (`performanceCategory` A best to E) quality.

In [ ]:
elements = [ELEMENTS[v]["elementId"] for v in enabled_variables(cfg) if v in ELEMENTS]
print("required elements:", elements)
req = available_series(None, src["station_id"].tolist(), elements)
print(len(req), "available series rows")
req.head()

In [ ]:
sel = select_stations(src, 60.39, st.get("bbox"), int(st["max_n"]), required=req,
                      exposure_max=st.get("exposure_max"), performance_max=st.get("performance_max"),
                      mountain_min_elev_m=float(st["classes"]["mountain_min_elev_m"]))
out = freeze(sel, cfg.get_path("stations_csv"))
print(len(sel), "stations frozen to", out)
sel

### Station class needs a hand edit

`select_stations` fills `station_class` from elevation only: `mountain` at or above 600 m, else `lowland`. Frost carries no coastline distance, so the split of lowland sites into `coastal` and `inland` (the `coastal_max_dist_km: 5` rule in the config) has to be made by hand in the frozen CSV, from a map or a coastline dataset. Do that once, commit the CSV, and the `station_class` stratum in the scores becomes meaningful. Until then the stratum shows `lowland` and `mountain` only.

In [ ]:
from afv.stations import load_frozen

stations = load_frozen(cfg.get_path("stations_csv"))
print(stations["station_class"].value_counts())
print(stations["county"].value_counts())
stations

## Element ids

The next cell uses the network: one Frost `/elements` request. `ELEMENTS` in `afv.fetch_obs` maps each variable to an element id, time resolution and level. Three ids were written from the naming pattern rather than from an official example and are flagged until this lookup confirms them: `wind_speed`, `wind_from_direction`, `sum(precipitation_amount PT1H)`. Confirmed from official examples: `air_temperature`, `air_pressure_at_sea_level`.

In [ ]:
from afv.fetch_obs import resolve_elements, ELEMENTS

df = resolve_elements()
print(df.to_string(index=False))
print()
found = set(df["id"])
flagged = {"wind_speed", "wind_from_direction", "sum(precipitation_amount PT1H)"}
for var, spec in ELEMENTS.items():
    eid = spec["elementId"]
    status = "ok" if eid in found else "NOT FOUND, fix ELEMENTS in src/afv/fetch_obs.py"
    note = " (was flagged)" if eid in flagged else ""
    print(f"{var:5s} {eid:35s} {status}{note}")

## Observations

The next cell uses the network: sequential Frost `/observations` requests, 10 stations and 2 days per request, cached as JSON under `cache_dir/frost_raw`. The window is one day before the first initialisation to the longest lead plus a day after the last one, so that persistence has an observation at every init time. Frost returns 403 for an over-large request; if that happens, lower `chunk_days` or `stations_per_request`.

In [ ]:
from afv.fetch_obs import fetch_observations, save_obs

runs = cfg["runs"]
t0 = pd.Timestamp(runs["window_start"], tz="UTC") - pd.Timedelta(days=1)
t1 = pd.Timestamp(runs["window_end"], tz="UTC") + pd.Timedelta(hours=max(runs["lead_hours"]) + 24)
station_ids = stations["station_id"].tolist()
cache_dir = cfg.get_path("cache_dir")
obs = fetch_observations(None, station_ids, ["t2m", "ws10", "msl"], t0, t1, cache_dir, qc_accept={0, 1})
print(len(obs), "rows |", obs["station_id"].nunique(), "stations |", t0, "to", t1)
obs.head()

### Quality codes before filtering

`fetch_observations` drops codes 6 (model data) and 7 outright and keeps everything else with the raw `qc` plus a boolean `qc_accepted`. The table below is the count per variable and code, so the effect of the accept list `{0, 1}` is visible before it is applied.

In [ ]:
counts = obs.groupby(["variable", "qc"]).size().unstack(fill_value=0)
print(counts)
print()
print("accepted share per variable:")
print(obs.groupby("variable")["qc_accepted"].mean().round(3))

In [ ]:
out = save_obs(obs, cfg.get_path("obs_dir") / "obs.parquet")
print(out)

## Match

No network from here on. This mirrors `afv match`: only accepted observations go in, every field file under the configured models is matched, baselines are appended as pseudo-models, and the pairwise-complete rule is applied before writing.

In [ ]:
import json

from afv.match import match_all, pairwise_complete
from afv.baselines import persistence, climatology
from afv.dataset import write_dataset

obs_acc = obs[obs["qc"].isin(set(cfg["qc"]["accept_codes"]))]
fields = sorted(Path(cfg.get_path("fields_dir")).glob("*.nc"))
print(len(fields), "field files;", len(obs_acc), "accepted observation rows")

matched = match_all(fields, stations, obs_acc, cfg)
print(len(matched), "matched rows before baselines")
parts = [matched]
if cfg["baselines"].get("persistence"):
    parts.append(persistence(matched, obs_acc))
if cfg["baselines"].get("climatology"):
    parts.append(climatology(matched, obs_acc))
full = pd.concat(parts, ignore_index=True)
if cfg["missing"].get("pairwise_complete", True):
    n_before = len(full)
    full = pairwise_complete(full, cfg["models"])
    print(f"pairwise complete: {n_before} -> {len(full)} rows")

out_dir = cfg.get_path("results_dir")
manifest = write_dataset(full, out_dir / "matched.parquet", cfg)
print(json.dumps(manifest, indent=2))

In [ ]:
print(full.groupby(["model", "variable", "interp"]).size().unstack("interp", fill_value=0))
full.head(10)

## The choices baked into the matched table

**Lapse correction.** For `t2m` only: `fcst = fcst_raw - gamma * (station_elev - model_elev)` with `gamma = 0.0065 K/m`, so a station above the model surface gets a colder forecast. Open data carries no orography, so `model_elev` comes from `paths.orography_nc`, a MEPS surface geopotential area-averaged to 0.25 degree (`afv.fetch_meps.orography_to_grid`). If that file is absent the correction is off and `lapse_corrected` is False; `fcst_raw` is always kept.

**Interpolation.** Both `nearest` and `bilinear` are computed for every row and stored in `interp`. Scores use the primary method per variable (`bilinear` for `t2m`, `ws10`, `msl`; `nearest` for accumulated `tp24`), and the other method appears as its own stratum so the difference stays visible.

**Time.** Everything is UTC. A Frost `referenceTime` is the end of the aggregation window, so an hourly value stamped 12:00 covers 11:00 to 12:00 and is compared with the forecast valid at 12:00. Matching is on the exact valid time unless `time.match_tolerance_min` is set.

**Pairwise complete.** A (station, variable, init, lead, interp) sample is kept only if every model in `cfg.models` has a forecast and the observation is present. Baseline rows ride along with the kept keys. Models are never compared on different cases.